# Ditto entity-matching exploration

Exploration only: this notebook is not imported by application code and never starts training implicitly. It inspects the generated pair splits and, when a trained artifact exists, evaluates it on held-out pairs.

In [ ]:
import json
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
parents = (cwd, *cwd.parents)
repo_root = next(
    (parent for parent in parents if (parent / 'data/processed').exists()),
    cwd,
)
split_dir = repo_root / 'data/processed/sih_product/splits'
artifact_dir = repo_root / 'ml/artifacts/ditto'
test_path = split_dir / 'pairs_test.csv'
print(
    'test split:', test_path.exists(),
    'trained artifact:', (artifact_dir / 'metadata.json').exists(),
)

In [ ]:
if test_path.exists():
    pairs = pd.read_csv(test_path)
    display(pairs['label'].value_counts(normalize=True).rename('fraction'))
    display(pairs[['left_record_id', 'right_record_id', 'label']].head())
else:
    print('Generate the documented synthetic data splits before running this cell.')

In [ ]:
if test_path.exists() and (artifact_dir / 'metadata.json').exists():
    from sklearn.metrics import classification_report, roc_auc_score

    from ml.entity_matching.infer import load, predict_proba

    model = load(artifact_dir)
    pairs = pd.read_csv(test_path)
    scores = [
        predict_proba(model, json.loads(a), json.loads(b))
        for a, b in zip(pairs.left, pairs.right, strict=True)
    ]
    truth = (pairs.label == 'match').to_numpy()
    report = classification_report(
        truth, [score >= 0.5 for score in scores], zero_division=0, output_dict=True
    )
    display(report)
    print('ROC AUC:', roc_auc_score(truth, scores))
else:
    print(
        'Train using ml/entity_matching/configs/ditto.yaml before evaluating an artifact.'
    )